In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from dotenv import find_dotenv

ROOT = Path(find_dotenv()).parent
df = pd.read_csv(ROOT / "data" / "processed" / "pm25_hourly_clean.csv")
df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_convert("Asia/Kolkata")

HORIZON = 24   # predict 24 hours ahead
results = []

for station, g in df.groupby("station"):
    s = g.sort_values("time").set_index("time")["pm25"]

    data = pd.DataFrame({
        "now": s,                      # PM2.5 right now
        "target": s.shift(-HORIZON),   # PM2.5 24 hours later
    }).dropna()

    split = int(len(data) * 0.8)       # first 80% = past, last 20% = future
    test = data.iloc[split:]

    error = test["now"] - test["target"]
    results.append({
        "station": station,
        "test_hours": len(test),
        "avg_pm25": round(test["target"].mean(), 1),
        "MAE": round(error.abs().mean(), 1),
        "RMSE": round(np.sqrt((error ** 2).mean()), 1),
    })

pd.DataFrame(results)

,station,test_hours,avg_pm25,MAE,RMSE
0,"Anand Vihar, New Delhi - DPCC",1520,61.4,31.0,41.3
1,New Delhi,1395,36.3,14.6,19.4
2,"Punjabi Bagh, Delhi - DPCC",1549,45.8,23.4,31.5
3,"R K Puram, Delhi - DPCC",1540,48.4,21.6,28.9
